##  Video-Based Soft Skill Integration

In this step, I integrate **video-based soft skill signals** into the CV–Job matching pipeline to enrich soft skill evaluation.

---

### 🔹 Loading Datasets
Two datasets are loaded:
- **CV–Job matching dataset (`df_matches`)**  
  Contains pairwise matching features between CVs and job descriptions.
- **Video soft skill dataset (`df_video`)**  
  Contains soft skill scores extracted from candidate video interviews.

Basic shape checks are performed to validate data integrity.

---

### 🔹 Video Soft Skills
The video dataset includes the following evaluated soft skills:

- Professionalism  
- Confidence  
- Engagement  
- Enthusiasm  
- Communication  
- Presentation  
- Clarity of Thought  
- Positivity  

These signals provide behavioral insights that are not available from text-based CV analysis alone.

---

### 🔹 Practical Score Normalization
To reduce the impact of outliers, I compute the **95th percentile (quantile)** for each video-based soft skill.

This value is used as a **practical maximum**, ensuring that extremely high scores do not disproportionately influence downstream features.

---

### 🔹 Job-to-Video Skill Mapping
A mapping is defined between **job-required soft skills** and **corresponding video-based attributes**.

Each job soft skill is associated with one or more video signals that best reflect it.  
For example:
- *Communication* → Communication  
- *Leadership* → Confidence, Communication, Professionalism  
- *Teamwork* → Engagement, Positivity  

This mapping enables alignment between:
- Text-derived job requirements
- Behavioral signals inferred from video data

---

### Purpose
This step bridges **text-based skill requirements** with **behavioral video signals**, allowing the model to capture a more holistic representation of candidate suitability.


In [2]:
# df_matches: your matching dataset
df_matches = pd.read_csv("cv_jd_matching_features (1).csv", engine='python', on_bad_lines='skip')

# df_video: video-based soft skill scores (32 rows)
df_video = pd.read_csv("original_soft_skills_labels (1).csv")

print(df_matches.shape)
print(df_video.shape)

(10176, 11)
(32, 9)


In [3]:
df_matches.head()

,cv_id,jd_id,matched_weight,required_weight,match_ratio,missing_required_skills_count,missing_hard_skills,missing_soft_skills,required_hard_skills,required_soft_skills,required_skill_count
0,1,1,1,1,1.000000,0,[],['adaptability'],['ai'],"['teamwork', 'leadership', 'customer focus', '...",5
1,1,2,3,4,0.750000,1,['systems engineering'],"['communication', 'problem solving']","['ai', 'systems engineering', 'financial analy...","['communication', 'problem solving']",6
2,1,3,1,3,0.333333,2,"['marketing', 'microsoft office']",[],"['ai', 'marketing', 'microsoft office']","['teamwork', 'leadership', 'growth mindset']",6
3,1,4,1,3,0.333333,2,"['sales', 'product management']","['problem solving', 'time management', 'work e...","['ai', 'sales', 'product management']","['problem solving', 'leadership', 'time manage...",9
4,1,5,5,8,0.625000,3,"['networking', 'sales', 'product management']","['problem solving', 'adaptability', 'attention...","['ai', 'models', 'reporting', 'networking', 's...","['teamwork', 'problem solving', 'leadership', ...",18


In [4]:
df_video.head()

,filename,Professionalism,Confidence,Engagement,Enthusiasm,Communication,Presentation,Clarity_of_Thought,Positivity
0,1,22.325152,24.274055,27.667313,24.350524,25.118650,31.367272,10.000000,26.734732
1,2,26.742030,29.641204,21.528136,18.778412,33.529725,34.581048,19.740825,18.163977
2,3,22.231299,26.236370,24.813414,20.888218,28.874650,34.537766,13.151371,20.004393
3,4,30.505533,26.801223,22.152171,19.188237,33.624323,36.044581,16.597973,26.780299
4,5,16.127129,17.002374,23.144053,23.599777,26.588006,20.296608,22.762280,20.531991


In [5]:
JOB_SKILL_MAPPING = {
    "communication": ["Communication"],
    "leadership": ["Confidence", "Communication", "Professionalism"],
    "teamwork": ["Engagement", "Positivity"],
    "problem solving": ["Clarity_of_Thought", "Communication"],
    "customer focus": ["Communication", "Positivity"],
    "adaptability": ["Enthusiasm", "Engagement"],
    "attention to detail": ["Professionalism", "Clarity_of_Thought"],
    "organizational skills": ["Professionalism", "Presentation"],
    "analytical thinking": ["Clarity_of_Thought"],
    "work ethic": ["Professionalism", "Enthusiasm"],
    "collaboration": ["Engagement", "Communication"],
    "growth mindset": ["Enthusiasm", "Confidence"],
    "time management": ["Professionalism", "Presentation"]
}

## Video-Based Soft Skill Normalization & Aggregation

This step processes raw video-based soft skill scores and aligns them with job-level soft skill requirements.

---

### 🔹 Practical Normalization
To reduce the effect of extreme values, each video-based soft skill is normalized using its **95th percentile** as a practical maximum.

- Scores are divided by the 95th percentile value
- Values are clipped to a maximum of `1.0`
- Final scores are scaled to a **0–100 range**

This produces robust and comparable soft skill scores across candidates.

---

### 🔹 Normalized Video Features
The normalized dataset (`df_video_norm`) contains standardized soft skill scores such as:
- Professionalism
- Confidence
- Engagement
- Communication
- Positivity  
and others, all on a consistent scale.

---

### 🔹 Job-Level Soft Skill Aggregation
Using a predefined job-to-video skill mapping:
- Each job-required soft skill is mapped to one or more video-based attributes
- The final score for each job skill is computed as the **mean** of its mapped video skills

This results in a new dataset (`df_job_skills`) where:
- Each row represents a CV
- Each column represents a job-aligned soft skill score derived from video signals

---

###  Outcome
This process transforms raw video interview signals into **job-relevant soft skill features**, enabling seamless integration with CV–Job matching and downstream ranking models.


In [6]:
VIDEO_SKILLS = [
    "Professionalism", "Confidence", "Engagement", "Enthusiasm",
    "Communication", "Presentation", "Clarity_of_Thought", "Positivity"
]

practical_max = df_video[VIDEO_SKILLS].quantile(0.95)

practical_max

,0.95
Professionalism,34.711474
Confidence,35.984695
Engagement,34.598194
Enthusiasm,28.972616
Communication,39.145136
Presentation,37.095452
Clarity_of_Thought,26.289479
Positivity,28.987613


In [7]:
df_video_norm = df_video.copy()

for skill in VIDEO_SKILLS:
    df_video_norm[skill] = (
        df_video_norm[skill] / practical_max[skill]
    ).clip(upper=1.0) * 100

df_video_norm.head(20)

,filename,Professionalism,Confidence,Engagement,Enthusiasm,Communication,Presentation,Clarity_of_Thought,Positivity
0,1,64.316347,67.456611,79.967508,84.046687,64.167998,84.558270,38.038030,92.228125
1,2,77.040895,82.371698,62.223295,64.814348,85.654894,93.221799,75.090208,62.661168
2,3,64.045967,72.909804,71.718813,72.096416,73.763059,93.105123,50.025226,69.010143
3,4,87.883141,74.479508,64.026957,66.228874,85.896554,97.167117,63.135421,92.385319
4,5,46.460514,47.248905,66.893819,81.455457,67.921608,54.714547,86.583229,70.830224
5,6,100.000000,92.886366,65.287212,64.243647,96.738754,100.000000,89.099775,91.312713
6,7,28.808918,27.789592,28.903243,34.515351,25.545958,26.957483,38.038030,34.497494
7,8,57.616885,65.765306,47.923531,51.596067,70.693847,84.545717,40.987229,37.112060
8,9,72.635710,69.316710,71.116604,72.896334,85.786977,86.706180,84.624829,86.103724
9,10,87.798900,79.245720,74.605248,81.454497,90.947632,97.396495,74.582580,95.520949


In [8]:
job_skill_rows = []

for _, row in df_video_norm.iterrows():
    cv_id = row["filename"]
    job_skill_scores = {"cv_id": cv_id}

    for job_skill, video_skills in JOB_SKILL_MAPPING.items():
        job_skill_scores[job_skill] = row[video_skills].mean()

    job_skill_rows.append(job_skill_scores)

df_job_skills = pd.DataFrame(job_skill_rows)

df_job_skills.head(3)

,cv_id,communication,leadership,teamwork,problem solving,customer focus,adaptability,attention to detail,organizational skills,analytical thinking,work ethic,collaboration,growth mindset,time management
0,1.0,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.038030,74.181517,72.067753,75.751649,74.437308
1,2.0,85.654894,81.689162,62.442231,80.372551,74.158031,63.518821,76.065552,85.131347,75.090208,70.927621,73.939094,73.593023,85.131347
2,3.0,73.763059,70.239610,70.364478,61.894143,71.386601,71.907614,57.035597,78.575545,50.025226,68.071192,72.740936,72.503110,78.575545


##  Soft Skill Matching with Video Signals

In this step, video-based soft skill scores are integrated with the CV–Job matching dataset to compute **job-aware soft skill alignment features**.

---

### 🔹 Merging Video Features
The CV–Job matching dataset is merged with the video-derived soft skill scores using `cv_id`.

This enriches each CV–JD pair with behavioral soft skill signals extracted from video interviews.

---

### 🔹 Soft Skill Match Score
For each CV–Job pair:
- The list of **required soft skills** for the job is retrieved
- Each required skill is mapped to its corresponding video-based score
- Scores are normalized to `[0, 1]` and averaged

The resulting feature:
- `soft_skill_match_score`  
represents how well a candidate satisfies the job’s soft skill requirements based on video evidence.

---

### 🔹 Missing Soft Skills (Threshold-Based)
To identify gaps in soft skills:
- A fixed threshold (`SKILL_THRESHOLD = 60`) is applied
- Any required soft skill with a score below the threshold is considered **missing**

Two additional features are generated:
- `missing_soft_skills_new`: list of missing soft skills
- `missing_soft_skills_count_new`: number of missing soft skills

This provides interpretable indicators of **behavioral skill gaps**.

---

### 🔹 Feature Comparison
The newly computed missing soft skills are compared with:
- Text-based missing soft skills extracted earlier

This allows analysis of differences between:
- Claimed or inferred skills from text
- Observed skills from video behavior

---

### 🔹 Final Feature Set
The final dataset includes:
- CV and Job identifiers
- Soft skill match score
- Lists and counts of missing soft skills

These features are appended back to the main matching dataset and used as **model-ready inputs** for downstream ranking or classification models.

---

###  Outcome
This step combines **text-based requirements** with **video-based behavioral evidence**, producing richer and more reliable soft skill matching signals.


In [9]:
df = df_matches.merge(df_job_skills, on="cv_id", how="left")

df.head(10)

,cv_id,jd_id,matched_weight,required_weight,match_ratio,missing_required_skills_count,missing_hard_skills,missing_soft_skills,required_hard_skills,required_soft_skills,required_skill_count,communication,leadership,teamwork,problem solving,customer focus,adaptability,attention to detail,organizational skills,analytical thinking,work ethic,collaboration,growth mindset,time management
0,1,1,1,1,1.000000,0,[],['adaptability'],['ai'],"['teamwork', 'leadership', 'customer focus', '...",5,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
1,1,2,3,4,0.750000,1,['systems engineering'],"['communication', 'problem solving']","['ai', 'systems engineering', 'financial analy...","['communication', 'problem solving']",6,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
2,1,3,1,3,0.333333,2,"['marketing', 'microsoft office']",[],"['ai', 'marketing', 'microsoft office']","['teamwork', 'leadership', 'growth mindset']",6,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
3,1,4,1,3,0.333333,2,"['sales', 'product management']","['problem solving', 'time management', 'work e...","['ai', 'sales', 'product management']","['problem solving', 'leadership', 'time manage...",9,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
4,1,5,5,8,0.625000,3,"['networking', 'sales', 'product management']","['problem solving', 'adaptability', 'attention...","['ai', 'models', 'reporting', 'networking', 's...","['teamwork', 'problem solving', 'leadership', ...",18,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
5,1,6,5,11,0.454545,6,"['engineering', 'systems engineering', 'accoun...",['time management'],"['data analysis', 'ai', 'models', 'engineering...","['teamwork', 'leadership', 'time management', ...",18,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
6,1,7,5,8,0.625000,3,"['systems engineering', 'operations management...","['problem solving', 'time management', 'adapta...","['ai', 'reporting', 'systems engineering', 'fi...","['teamwork', 'problem solving', 'leadership', ...",19,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
7,1,8,3,7,0.428571,4,"['networking', 'sales', 'product management', ...","['problem solving', 'time management', 'adapta...","['ai', 'reporting', 'networking', 'financial a...","['teamwork', 'problem solving', 'leadership', ...",18,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
8,1,9,1,3,0.333333,2,"['systems engineering', 'sales']",[],"['systems engineering', 'financial analysis', ...","['leadership', 'customer focus', 'growth minds...",6,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308
9,1,10,4,8,0.500000,4,"['engineering', 'networking', 'product managem...","['time management', 'adaptability', 'attention...","['ai', 'reporting', 'engineering', 'networking...","['teamwork', 'leadership', 'time management', ...",18,64.167998,65.313652,86.097817,51.103014,78.198062,82.007097,51.177188,74.437308,38.03803,74.181517,72.067753,75.751649,74.437308


In [10]:
def compute_match_score(row):
    required = row["required_soft_skills"]

    if not required or len(required) == 0:
        return 0.0

    scores = [row[skill] / 100 for skill in required]
    return np.mean(scores)

df['required_soft_skills'] = df['required_soft_skills'].apply(ast.literal_eval)

df["soft_skill_match_score"] = df.apply(compute_match_score, axis=1)

df[["cv_id", "jd_id", "soft_skill_match_score"]].head()

,cv_id,jd_id,soft_skill_match_score
0,1,1,0.779042
1,1,2,0.576355
2,1,3,0.757210
3,1,4,0.698309
4,1,5,0.710335


In [ ]:
SKILL_THRESHOLD = 60

In [ ]:
def find_missing_soft_skills(row):
    missing = []
    for skill in row["required_soft_skills"]:
        if row[skill] < SKILL_THRESHOLD:
            missing.append(skill)
    return missing


df["missing_soft_skills_new"] = df.apply(find_missing_soft_skills, axis=1)
df["missing_soft_skills_count_new"] = df["missing_soft_skills_new"].apply(len)

df[
    ["cv_id", "jd_id", "soft_skill_match_score",
     "missing_soft_skills_new", "missing_soft_skills_count_new"]
].head()

,cv_id,jd_id,soft_skill_match_score,missing_soft_skills_new,missing_soft_skills_count_new
0,1,1,0.779042,[],0
1,1,2,0.576355,[problem solving],1
2,1,3,0.757210,[],0
3,1,4,0.698309,[problem solving],1
4,1,5,0.710335,"[problem solving, attention to detail]",2


In [ ]:
df[["soft_skill_match_score", "missing_soft_skills_count_new"]].describe()

,soft_skill_match_score,missing_soft_skills_count_new
count,4625.000000,4625.000000
mean,0.609449,2.767135
std,0.202963,3.264779
min,0.000000,0.000000
25%,0.408511,0.000000
50%,0.666368,1.000000
75%,0.776453,5.000000
max,0.982709,12.000000


In [ ]:
df[["missing_soft_skills", "missing_soft_skills_new"]].head()

,missing_soft_skills,missing_soft_skills_new
0,['adaptability'],[]
1,"['communication', 'problem solving']",[problem solving]
2,[],[]
3,"['problem solving', 'time management', 'work e...",[problem solving]
4,"['problem solving', 'adaptability', 'attention...","[problem solving, attention to detail]"


In [ ]:
final_cols = [
    "cv_id", "jd_id",
    "missing_soft_skills",
    "required_soft_skills",
    "soft_skill_match_score",
    "missing_soft_skills_new",
    "missing_soft_skills_count_new"
]

df_final = df[final_cols]

df_final.head(10)

,cv_id,jd_id,missing_soft_skills,required_soft_skills,soft_skill_match_score,missing_soft_skills_new,missing_soft_skills_count_new
0,1,1,['adaptability'],"[teamwork, leadership, customer focus, adaptab...",0.779042,[],0
1,1,2,"['communication', 'problem solving']","[communication, problem solving]",0.576355,[problem solving],1
2,1,3,[],"[teamwork, leadership, growth mindset]",0.757210,[],0
3,1,4,"['problem solving', 'time management', 'work e...","[problem solving, leadership, time management,...",0.698309,[problem solving],1
4,1,5,"['problem solving', 'adaptability', 'attention...","[teamwork, problem solving, leadership, custom...",0.710335,"[problem solving, attention to detail]",2
5,1,6,['time management'],"[teamwork, leadership, time management, organi...",0.694491,[analytical thinking],1
6,1,7,"['problem solving', 'time management', 'adapta...","[teamwork, problem solving, leadership, time m...",0.680572,"[problem solving, attention to detail, analyti...",3
7,1,8,"['problem solving', 'time management', 'adapta...","[teamwork, problem solving, leadership, time m...",0.713429,"[problem solving, attention to detail]",2
8,1,9,[],"[leadership, customer focus, growth mindset]",0.730878,[],0
9,1,10,"['time management', 'adaptability', 'attention...","[teamwork, leadership, time management, custom...",0.733669,[attention to detail],1


In [ ]:
new_cols = [
    "soft_skill_match_score",
    "missing_soft_skills_new",
    "missing_soft_skills_count_new"
]

df_final = df_matches.copy()

for col in new_cols:
    df_final[col] = df[col].values

df_final.head()

,cv_id,jd_id,matched_weight,required_weight,match_ratio,missing_required_skills_count,missing_hard_skills,missing_soft_skills,required_hard_skills,required_soft_skills,required_skill_count,soft_skill_match_score,missing_soft_skills_new,missing_soft_skills_count_new
0,1,1,1,1,1.000000,0,[],['adaptability'],['ai'],"['teamwork', 'leadership', 'customer focus', '...",5,0.779042,[],0
1,1,2,3,4,0.750000,1,['systems engineering'],"['communication', 'problem solving']","['ai', 'systems engineering', 'financial analy...","['communication', 'problem solving']",6,0.576355,[problem solving],1
2,1,3,1,3,0.333333,2,"['marketing', 'microsoft office']",[],"['ai', 'marketing', 'microsoft office']","['teamwork', 'leadership', 'growth mindset']",6,0.757210,[],0
3,1,4,1,3,0.333333,2,"['sales', 'product management']","['problem solving', 'time management', 'work e...","['ai', 'sales', 'product management']","['problem solving', 'leadership', 'time manage...",9,0.698309,[problem solving],1
4,1,5,5,8,0.625000,3,"['networking', 'sales', 'product management']","['problem solving', 'adaptability', 'attention...","['ai', 'models', 'reporting', 'networking', 's...","['teamwork', 'problem solving', 'leadership', ...",18,0.710335,"[problem solving, attention to detail]",2


## Final Scoring, Domain Weighting & Target Label Generation

In this step, all engineered features are combined to compute a **final matching score** and generate the binary target label.

---

### 🔹 Dataset Inspection
Basic exploration is performed on the newly added soft-skill features using summary statistics and row-level inspection to ensure correctness and reasonable distributions.

---

### 🔹 Domain Integration
Each job description is associated with a **domain** (e.g., data, engineering, business):

- A mapping between `jd_id` and `domain` is created from an external job dataset
- The domain information is merged into the matching dataset
- This allows domain-specific weighting in the final scoring step

---

### 🔹 Domain-Aware Weighting
A helper function retrieves **domain-specific weights** for hard and soft skills.

- If a domain is not found, default weights are used:
  - Hard skills: 50%
  - Soft skills: 50%

This enables flexible scoring strategies across different job domains.

---

### 🔹 Final Match Score
For each CV–Job pair, a final score is computed as:



In [ ]:
df_final[new_cols].describe()

,soft_skill_match_score,missing_soft_skills_count_new
count,4625.000000,4625.000000
mean,0.609449,2.767135
std,0.202963,3.264779
min,0.000000,0.000000
25%,0.408511,0.000000
50%,0.666368,1.000000
75%,0.776453,5.000000
max,0.982709,12.000000


In [ ]:
df_final.loc[0, :]

,0
cv_id,1
jd_id,1
matched_weight,1
required_weight,1
match_ratio,1.0
missing_required_skills_count,0
missing_hard_skills,[]
missing_soft_skills,['adaptability']
required_hard_skills,['ai']
required_soft_skills,"['teamwork', 'leadership', 'customer focus', '..."


In [ ]:
df_final.to_csv("final_dataset.csv", index=False)

# Adding Final Label

In [ ]:
df2 = pd.read_csv('Final_JDs.csv')

# Create a dictionary mapping jd_id to domain
jd_domain_map = df2.set_index('jd_id')['domain'].to_dict()

# Add the domain column to the first dataset
df_final['domain'] = df_final['jd_id'].map(jd_domain_map)

# Save the updated dataset
df_final.to_csv('updated_dataset.csv', index=False)

df_final.columns

Index(['cv_id', 'jd_id', 'matched_weight', 'required_weight', 'match_ratio', 'missing_required_skills_count',
       'missing_hard_skills', 'missing_soft_skills', 'required_hard_skills', 'required_soft_skills',
       'required_skill_count', 'soft_skill_match_score', 'missing_soft_skills_new', 'missing_soft_skills_count_new',
       'domain'],
      dtype='object')

In [ ]:
df_final.head(10)

,cv_id,jd_id,matched_weight,required_weight,match_ratio,missing_required_skills_count,missing_hard_skills,missing_soft_skills,required_hard_skills,required_soft_skills,required_skill_count,soft_skill_match_score,missing_soft_skills_new,missing_soft_skills_count_new,domain
0,1,1,1,1,1.000000,0,[],['adaptability'],['ai'],"['teamwork', 'leadership', 'customer focus', '...",5,0.779042,[],0,Support
1,1,2,3,4,0.750000,1,['systems engineering'],"['communication', 'problem solving']","['ai', 'systems engineering', 'financial analy...","['communication', 'problem solving']",6,0.576355,[problem solving],1,IT
2,1,3,1,3,0.333333,2,"['marketing', 'microsoft office']",[],"['ai', 'marketing', 'microsoft office']","['teamwork', 'leadership', 'growth mindset']",6,0.757210,[],0,Business
3,1,4,1,3,0.333333,2,"['sales', 'product management']","['problem solving', 'time management', 'work e...","['ai', 'sales', 'product management']","['problem solving', 'leadership', 'time manage...",9,0.698309,[problem solving],1,Business
4,1,5,5,8,0.625000,3,"['networking', 'sales', 'product management']","['problem solving', 'adaptability', 'attention...","['ai', 'models', 'reporting', 'networking', 's...","['teamwork', 'problem solving', 'leadership', ...",18,0.710335,"[problem solving, attention to detail]",2,Business
5,1,6,5,11,0.454545,6,"['engineering', 'systems engineering', 'accoun...",['time management'],"['data analysis', 'ai', 'models', 'engineering...","['teamwork', 'leadership', 'time management', ...",18,0.694491,[analytical thinking],1,Finance
6,1,7,5,8,0.625000,3,"['systems engineering', 'operations management...","['problem solving', 'time management', 'adapta...","['ai', 'reporting', 'systems engineering', 'fi...","['teamwork', 'problem solving', 'leadership', ...",19,0.680572,"[problem solving, attention to detail, analyti...",3,Engineering
7,1,8,3,7,0.428571,4,"['networking', 'sales', 'product management', ...","['problem solving', 'time management', 'adapta...","['ai', 'reporting', 'networking', 'financial a...","['teamwork', 'problem solving', 'leadership', ...",18,0.713429,"[problem solving, attention to detail]",2,Finance
8,1,9,1,3,0.333333,2,"['systems engineering', 'sales']",[],"['systems engineering', 'financial analysis', ...","['leadership', 'customer focus', 'growth minds...",6,0.730878,[],0,Business
9,1,10,4,8,0.500000,4,"['engineering', 'networking', 'product managem...","['time management', 'adaptability', 'attention...","['ai', 'reporting', 'engineering', 'networking...","['teamwork', 'leadership', 'time management', ...",18,0.733669,[attention to detail],1,Administrative


In [ ]:
# get domain distinct
df_final['domain'].unique()

array(['Support', 'IT', 'Business', 'Finance', 'Engineering',
       'Administrative', 'Creative', 'HR', 'Research', 'Operations',
       'Management', 'Consulting', 'Analytics'], dtype=object)

In [ ]:
DOMAIN_WEIGHTS = {
    "Engineering":     {"hard": 0.70, "soft": 0.30},
    "IT":              {"hard": 0.70, "soft": 0.30},
    "Analytics":       {"hard": 0.75, "soft": 0.25},

    "Finance":         {"hard": 0.65, "soft": 0.35},
    "Research":        {"hard": 0.65, "soft": 0.35},

    "Business":        {"hard": 0.50, "soft": 0.50},
    "Operations":      {"hard": 0.50, "soft": 0.50},

    "Management":     {"hard": 0.45, "soft": 0.55},
    "Consulting":     {"hard": 0.45, "soft": 0.55},

    "Administrative": {"hard": 0.40, "soft": 0.60},

    "Support":         {"hard": 0.30, "soft": 0.70},
    "HR":              {"hard": 0.30, "soft": 0.70},

    "Creative":        {"hard": 0.25, "soft": 0.75}
}

In [ ]:
def get_domain_weights(domain):
    if isinstance(domain, list) and len(domain) > 0:
        domain = domain[0]

    return DOMAIN_WEIGHTS.get(
        domain,
        {"hard": 0.50, "soft": 0.50}
    )

In [ ]:
ACCEPT_THRESHOLD = 0.6

final_scores = []

for _, row in df_final.iterrows():
    weights = get_domain_weights(row["domain"])

    final_score = (
        weights["hard"] * row["match_ratio"] +
        weights["soft"] * row["soft_skill_match_score"]
    )

    final_scores.append(final_score)

df_final["final_score"] = final_scores
df_final["target"] = (df_final["final_score"] >= ACCEPT_THRESHOLD).astype(int)

df_final.head()

,cv_id,jd_id,matched_weight,required_weight,match_ratio,missing_required_skills_count,missing_hard_skills,missing_soft_skills,required_hard_skills,required_soft_skills,required_skill_count,soft_skill_match_score,missing_soft_skills_new,missing_soft_skills_count_new,domain,final_score,target
0,1,1,1,1,1.000000,0,[],['adaptability'],['ai'],"['teamwork', 'leadership', 'customer focus', '...",5,0.779042,[],0,Support,0.845329,1
1,1,2,3,4,0.750000,1,['systems engineering'],"['communication', 'problem solving']","['ai', 'systems engineering', 'financial analy...","['communication', 'problem solving']",6,0.576355,[problem solving],1,IT,0.697907,1
2,1,3,1,3,0.333333,2,"['marketing', 'microsoft office']",[],"['ai', 'marketing', 'microsoft office']","['teamwork', 'leadership', 'growth mindset']",6,0.757210,[],0,Business,0.545272,0
3,1,4,1,3,0.333333,2,"['sales', 'product management']","['problem solving', 'time management', 'work e...","['ai', 'sales', 'product management']","['problem solving', 'leadership', 'time manage...",9,0.698309,[problem solving],1,Business,0.515821,0
4,1,5,5,8,0.625000,3,"['networking', 'sales', 'product management']","['problem solving', 'adaptability', 'attention...","['ai', 'models', 'reporting', 'networking', 's...","['teamwork', 'problem solving', 'leadership', ...",18,0.710335,"[problem solving, attention to detail]",2,Business,0.667668,1


In [ ]:
df_final["target"].value_counts(normalize=True)

,proportion
target,
0,0.823351
1,0.176649


In [ ]:
#download final df
df_final.to_csv('final_df_bgd.csv', index=False)